# 09 · Synthesis: Cross-Check, SWOT, Ethics, Conclusion

The written record. No computation — every number quoted here comes from a stage above,
and the figures referenced are the exported PNGs in `docs/figures/`.

## 12. Full Cross-Check Summary: Blueprint vs. Thesis vs. Repo

| # | Item | Finding | Resolution |
|---|---|---|---|
| 1 | `Bae-Youn/eventstudy` citation | Fabricated — no such repo/author exists | Real package identified: `LemaireJean-Baptiste/eventstudy` (not needed here — see #3) |
| 2 | Blueprint's `.fillna(0)` for missing Volume | Contradicts thesis §3.4.1's forward-fill mandate | Thesis followed; blueprint rejected on this point |
| 3 | Y1 target formula | Initially mis-flagged as needing market-model AR/CAR (misread §2.2.3 as operational) | Corrected: §3.2.2 raw log return confirmed correct as originally implemented |
| 4 | K-fold CV (proposal §7.5) vs. walk-forward-only (thesis §3.7.1) | Internal conflict between the project's own two prior documents | Thesis version adopted (walk-forward only, no k-fold, anywhere) |
| 5 | Hyperparameter search | Built in repo (`optimizers.py`) but never called; blueprint hardcodes fixed values too; first working version only tuned RF against Y1 as a proxy for all 3 targets, and never searched XGBoost at all | Rebuilt in §9: `GridSearchCV` wraps the actual `MultiOutputRegressor` directly, scored with a custom weighted-RMSE scorer averaged across all 3 targets (thesis eq.(4) weights); XGBoost now has its own real grid too |
| 6 | SMOGN fidelity | Repo's version was Gaussian-noise-only, not true SmoteR-with-interpolation | Rewritten in `time_aware_smogn.py`: real neighbor-pair interpolation within a 5-year temporal window, jointly across X and all 3 targets. Real `nickkunz/smogn` package considered but rejected — single-response-variable API doesn't fit this project's joint 3-target setup |
| 7 | 30-day panic-proxy rolling std | Thesis §3.5.2 requires it; repo only had 5/10/20-day windows | Added (`rolling_std_30`) |
| 8 | SMA/EMA look-ahead | Repo computed rolling windows on unshifted price — a real bug, found independent of the blueprint | Fixed: rolling windows now shift price by 1 day first |
| 9 | PPP/CPI damage adjustment | Thesis §3.5.3 requires it; repo had no deflator step | Resolved via EM-DAT's own native CPI-adjusted damage column (verified against doc.emdat.be), not a separately-coded deflator |
| 10 | Macro/global controls | `build_feature_table()`'s merge params existed, never called with real data | Real World Bank + S&P 500 data wired in §2/§7 |
| 11 | CSE data source (Yahoo-only implied by blueprint) | Yahoo alone unreliable for CSE volume (index ticker); thesis mandates cross-verification | Resolved: real local archive is primary; Yahoo gap-fill attempted and found dead (§5), not used |
| 12 | Per-target evaluation | First working version's fold loop only ever scored Y1 (`pred[:, 0]` hardcoded) — Y2/Y3 were trained but never verified, a real gap found when directly asked "did it predict each output" | Rebuilt in §9/§10: every model scored on all 3 targets separately, 12-row summary table |
| 13 | Sector-level feature from `market-capitalization-Oct 12.csv` | Considered for §7's new features. File is a **single static date**, and its `Symbol` codes (e.g. `JKH.N0000`) don't match the numeric Company IDs used in the yearly per-security volume files — no real historical join key exists | Rejected: would require fabricating a constant, non-time-varying value into a time-varying model. Not built |
| 14 | New engineered features (§7) | Disaster recency (`days_since_last_disaster`, `disasters_trailing_365d`), damage-to-GDP ratio, one Flood-only interaction term | Added — all from real data already loaded, no new source needed |
| 15 | Feature-count growth vs. N | New features push `FEATURE_COLS` from 28 to ~32 against N≈64-74 — real overfitting risk | RF-importance top-20 feature selection added, fit per fold on training data only (§9) |
</cell id="dde97685">


## 13. SWOT Analysis

### Strengths
- Multi-task architecture captures price, volume, and recovery jointly — a stable index alone masks liquidity crises the way single-target models can't see (thesis §2.7 gap; Saberironaghi et al., 2025).
- Tree ensembles + a heavily-regularized shallow MLP is the scientifically defensible choice at N≈50-86 — LSTM/Transformer would memorize noise instead of generalizing (Venkatarathnam et al., 2024; Perera, 2025), and both the thesis and the blueprint independently reach this same exclusion.
- Chronological walk-forward + time-aware SMOGN structurally prevents the two classic financial-ML leakage failure modes.
- SHAP gives the regulatory transparency a CBSL/SEC-style body would require before accepting a black-box model (Dugbartey, 2025).
- **Real, verified data throughout** — every number in §2 traces to an inspected file or a live-checked API call, not an assumption.
- **All three targets verified separately, not just Y1** (§10) — a real gap found and fixed, not assumed away.
- Hyperparameter search now spans RF and XGBoost jointly, scored against all 3 targets per the thesis's own eq.(4) weighting, not a Y1-only proxy.

### Weaknesses
- N≈64 (in-scope, real) is still small even after SMOGN — synthetic augmentation cannot manufacture genuinely new tail-risk information.
- EM-DAT damage figures are sparse for Sri Lanka: 68/86 raw records have **no** damage estimate at all (§2 above), a known reporting-bias limitation (Caldera & Wirasinghe, 2022; thesis §3.9.1) that directly affects roughly half the study window.
- The archive's real coverage ends mid-2023 — the most recent ~2.5 years, including the highest-profile event in the thesis itself, are unmodeled.
- Hyperparameter search is grid-based (not Bayesian) — wider than the first working version, but still modest given N; a larger search would likely help further with more real data.
- New features (§7) increase the feature-to-N ratio; per-fold feature selection (§9) mitigates but doesn't eliminate this risk at N≈64-74.

### Opportunities
- First predictive (not retrospective) ML framework for the CSE / South Asian frontier markets — a gap the thesis's own literature review (§2.7) confirms is real and unaddressed.
- Direct policy-tool potential for CBSL/SEC Sri Lanka **ex-post impact assessment** (not early warning — see the feature-timing limitation in §15); SDG 8/9/11/13 alignment (proposal §4.4).
- The 64-event real training table, EM-DAT loader, and CSE archive parsers built here are all reusable beyond this one notebook.

### Threats
- Overclaiming risk: at this N, "beats the Ridge baseline on held-out RMSE" is a real, honest signal — it is not proof of production-grade predictive power, and should never be presented as such.
- Data-source fragility: the live-data dead-end found in §5 (a real, actively-listed ticker whose backend feed silently stopped updating years ago) shows that "the API exists" and "the API works" are different claims that must each be checked, not assumed, before any future extension of this model.
- Compound/overlapping disaster seasons (common in Sri Lanka's monsoon pattern) stress-test the event-window-truncation logic; the new `disasters_trailing_365d` feature (§7) makes this pattern visible to the model rather than just naming it as a risk.
</cell id="c868f959">


## 14. SMART Solution Statement

- **Specific** — a multi-target model (ASPI return, abnormal volume, recovery days) trained on real CSE archive data (2000–Jun/Mar-2023) and real EM-DAT disaster records, following the thesis's §3 methodology exactly where verified correct, and departing from the supplied blueprint only where the blueprint itself was shown to contradict the thesis or established data-leakage practice.
- **Measurable** — RMSE/MAE/R² per target vs. the Ridge baseline, walk-forward held-out (§10); headline result: Random Forest and XGBoost beat the Ridge baseline on Y1 RMSE in every fold tested (§10 table).
- **Achievable** — built entirely on the existing repo scaffold plus three new, tested loader modules; no architecture rewrite, no paid API, no fabricated data.
- **Relevant** — directly answers thesis RQ2/RQ3/RO3/RO4, and closes the literature gap the thesis's own §2.7 identifies.
- **Time-bound** — aligned to the proposal's own Gantt chart (§9): data pipeline and model now real and tested; remaining work (CBSL manual macro pull, any future live-data re-verification) explicitly scoped as follow-up, not blocking this deliverable.


## 15. Ethics, Reliability & Defensibility

**Not investment advice.** This model is an academic **ex-post impact-attribution** research artifact, **not an early-warning system**. Six of its features (`financial_damage`, `population_affected`, their log transforms, `damage_to_gdp`, and `log_damage_x_flood`) are EM-DAT damage assessments finalized weeks to months after an event. On the day before a flood, its eventual total damage and total affected are unknown, so the model as specified **cannot be run prospectively**. What it answers is the attribution question — *given a disaster of known severity, what was the market's response* — which is a legitimate research question and the one the results here actually address. An ex-ante variant (disaster type, season, historical type-average severity, trailing disaster count) is a separate experiment, recorded in §16 as future work. Earlier drafts of this notebook and the README described the artifact as an early-warning system; that framing was wrong and is corrected here. Its outputs should never be presented to, or used by, retail or institutional investors as a trading signal without independent validation far beyond what N≈64 events can support.

**Data provenance is explicit, not implicit.** Every row in the market series (§2) carries a `price_source`/`volume_source` column distinguishing real-local-archive data from anything else — currently, everything is real-local-archive, since the live gap-fill was verified dead and abandoned rather than silently substituted (§5).

**Reproducibility.** `RANDOM_STATE = 42` fixed throughout (§2 setup cell); `requirements.txt` pins all dependency floors; every external claim in §1.3 was checked live during this notebook's own development, with the checking method disclosed, not just the conclusion.

**Limitations (thesis §3.9 style, restated for this real-data run):**
- EM-DAT pre-2010 and general Sri Lankan reporting bias (Caldera & Wirasinghe, 2022) — most severe for smaller, non-catastrophic events.
- Frontier-market microstructure frictions (chronic illiquidity, thin trading) — thesis §3.9.2, directly visible in this archive's own volume figures.
- The Jul-2023 → 2025 gap (§5), Ditwah included, is real and unresolved as of this notebook — flagged as explicit future work, not hidden.
- CBSL's higher-frequency macro series (monthly CCPI, daily LKR/USD, policy rate) were not sourced — only World Bank's coarser annual GDP/inflation proxy was used (§1.2), a documented gap, not a silent omission.

**SDG alignment** (proposal §4.4): Climate Action (13), Decent Work & Economic Growth (8), Industry/Innovation/Infrastructure (9), Sustainable Cities & Communities (11) — via a data-driven impact-assessment tool for a climate-exposed frontier financial market.


## 16. Conclusion & Next Steps

This notebook implements the thesis's multi-target predictive framework end-to-end on real Colombo Stock Exchange and EM-DAT data — not a synthetic stand-in — and validates every external source and methodological claim it depends on against the thesis's own stated methodology, correcting its own earlier mistakes in the process (§5, §6, §9/§10) rather than presenting a falsely tidy narrative.

**Immediate next steps, in priority order:**
1. Source CBSL's monthly CCPI / daily LKR-USD / policy-rate series manually to replace the coarser World Bank annual proxy (§1.2, §15).
2. Investigate a working live or manual source for the Jul-2023 → 2025 window so Ditwah can eventually be evaluated, not just cited (§5).
3. Re-run hyperparameter search with a larger grid once/if more real data becomes available — the current grid was deliberately kept small for N≈64-74.
4. If Y2/Y3 RMSE (§10) proves consistently weaker than Y1's, investigate target-specific feature sets (per-target feature selection rather than one shared top-20 list) rather than assuming one feature set serves all three equally well.
</cell id="81f78e3b">
